# Kalshi DB Explorer

Interactive exploration of `~/.btc_windows.db` — Kalshi window settlements, sub-second tick paths, Polymarket asks, and Coinbase spot prices.

**Setup**: run the next cell to install the optional `itables` package for interactive DataFrame rendering. With it, every DataFrame becomes a sortable/searchable/paginated HTML table — no more column-truncation pain.

**Tables in the DB**:
- `windows` — one row per Kalshi 15m window (ticker, winner, floor strike, coin_open_price)
- `ticks` — Kalshi yes_ask / no_ask, **sub-second precision** (`elapsed_ms`)
- `poly_windows` / `poly_ticks` — Polymarket window + ask history
- `spot_ticks` — Coinbase crypto prices, sub-second on change

In [ ]:
# Install itables for nice interactive tables. Comment out if you already have it.
# !pip install -q itables

In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta, timezone

ET = timezone(timedelta(hours=-4))
DB = Path.home() / '.btc_windows.db'
con = sqlite3.connect(DB)

# ── Display config ─────────────────────────────────────────────────────────
# Try itables first (best interactive renderer). Fall back to widened pandas.
try:
    from itables import init_notebook_mode, show as _itable_show
    import itables.options as itopt
    init_notebook_mode(all_interactive=False)  # opt-in via show()
    itopt.maxBytes = 200_000_000          # large dataframes OK
    itopt.lengthMenu = [25, 50, 100, 250, 500]
    itopt.scrollX = True                  # horizontal scroll for wide tables
    itopt.classes = 'display compact nowrap'
    _ITABLES_OK = True
    print('itables loaded — call show(df) for interactive tables')
except ImportError:
    _ITABLES_OK = False
    print('itables not available — pip install itables for interactive tables')

# Widened pandas defaults (used as fallback)
pd.set_option('display.max_columns',     None)
pd.set_option('display.max_colwidth',    100)
pd.set_option('display.width',           200)
pd.set_option('display.max_rows',        100)
pd.set_option('display.float_format',    lambda v: f'{v:.4f}')

def show(df, n=None):
    """Display a DataFrame with the best available renderer.
    - itables: full interactive (sort/filter/paginate)
    - fallback: widened pandas display
    Pass n to limit rows."""
    if df is None or len(df) == 0:
        print('(empty result)')
        return
    sub = df.head(n) if n else df
    print(f'{len(df):,} rows × {len(df.columns)} cols')
    if _ITABLES_OK:
        return _itable_show(sub)
    return sub

print(f'Connected to {DB}')
print(f'  size: {DB.stat().st_size / 1e6:.1f} MB')

## Schema + row counts

In [ ]:
rows = []
for tbl in ['windows', 'ticks', 'poly_windows', 'poly_ticks', 'spot_ticks']:
    try:
        n = con.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    except sqlite3.OperationalError:
        n = None
    rows.append({'table': tbl, 'rows': n})

show(pd.DataFrame(rows))

In [ ]:
# Schema for each table
schema = []
for tbl in ['windows', 'ticks', 'poly_windows', 'poly_ticks', 'spot_ticks']:
    try:
        for cid, name, ctype, notnull, default, pk in con.execute(f'PRAGMA table_info({tbl})'):
            schema.append({'table': tbl, 'col': name, 'type': ctype,
                           'pk': bool(pk), 'notnull': bool(notnull)})
    except sqlite3.OperationalError:
        pass

show(pd.DataFrame(schema))

## Helper functions

Reusable query wrappers that return DataFrames with friendly columns + ET timestamps.

In [ ]:
def to_et(unix_ts):
    if pd.isna(unix_ts): return None
    return datetime.fromtimestamp(unix_ts, tz=ET).strftime('%a %m/%d %H:%M:%S')

def get_recent_windows(coin=None, limit=50):
    """Most recent N windows. Optionally filter by coin (BTC/ETH/SOL/XRP)."""
    sql = """SELECT id, ticker, window_start_ts, winner, floor_strike, coin_open_price
             FROM windows WHERE ticker LIKE 'KX%15M%' """
    params = ()
    if coin:
        sql += "AND ticker LIKE 'KX' || ? || '%' "
        params = (coin,)
    sql += 'ORDER BY id DESC LIMIT ?'
    params = (*params, limit)
    df = pd.read_sql(sql, con, params=params)
    df['et_open'] = df['window_start_ts'].apply(to_et)
    df['coin'] = df['ticker'].str[2:5]
    # Keep window_start_ts so downstream cells can use it for tick joins.
    return df[['id','et_open','coin','ticker','winner','floor_strike',
               'coin_open_price','window_start_ts']]


def get_ticks(window_id):
    """Sub-second Kalshi ask path for one window."""
    df = pd.read_sql("""
        SELECT elapsed_sec, elapsed_ms, yes_ask, no_ask
        FROM ticks WHERE window_id=? ORDER BY elapsed_ms
    """, con, params=(window_id,))
    df['t_sec'] = df['elapsed_ms'] / 1000
    df['spread'] = df['yes_ask'] + df['no_ask'] - 1.0
    return df


def get_spot(coin, since_minutes=15):
    """Coinbase spot prices for one coin in last N minutes."""
    cutoff = int((datetime.now().timestamp() - since_minutes*60) * 1000)
    df = pd.read_sql("""
        SELECT ts_ms, price, source
        FROM spot_ticks WHERE coin=? AND ts_ms > ? ORDER BY ts_ms
    """, con, params=(coin, cutoff))
    df['et_time'] = df['ts_ms'].apply(lambda ms: datetime.fromtimestamp(
        ms/1000, tz=ET).strftime('%H:%M:%S.%f')[:-3])
    df['coin'] = coin
    return df[['et_time','ts_ms','coin','price','source']]


def get_window_with_ticks(coin, et_hour=None, et_minute=None):
    """Find a window matching coin + ET hour:min, return its row + ticks.
    If hour/minute omitted, returns the most recent window's ticks."""
    if et_hour is None:
        win = get_recent_windows(coin=coin, limit=1)
    else:
        # Match by ET hour/minute on the ticker (close time embedded in name).
        # Ticker fmt: KXBTC15M-26APR272115-15  → close at 21:15 ET
        et_pat = (f'%{et_hour:02d}{et_minute:02d}-%' if et_minute is not None
                  else f'%{et_hour:02d}%-%')
        df = pd.read_sql("""
            SELECT id, ticker, window_start_ts, winner, floor_strike, coin_open_price
            FROM windows WHERE ticker LIKE 'KX' || ? || '15M%' AND ticker LIKE ?
            ORDER BY id DESC LIMIT 5
        """, con, params=(coin, et_pat))
        if len(df):
            df['et_open'] = df['window_start_ts'].apply(to_et)
            df['coin'] = df['ticker'].str[2:5]
        win = df
    if len(win) == 0:
        return None, None
    wid = int(win.iloc[0]['id'])
    return win.iloc[0], get_ticks(wid)


print('Helpers ready: get_recent_windows, get_ticks, get_spot, get_window_with_ticks')

## Examples

Each cell uses `show()` for interactive display when itables is available.

In [ ]:
# Last 50 windows across all coins
show(get_recent_windows(limit=50))

In [ ]:
# Just BTC, last 20 windows
show(get_recent_windows(coin='BTC', limit=20))

In [ ]:
# Sub-second tick path for the most recent BTC window
row, ticks = get_window_with_ticks('BTC')
print(f"Window {row['ticker']}  (started {to_et(row['window_start_ts'])})")
show(ticks, n=200)

In [ ]:
# Coinbase BTC spot prices, last 5 minutes
show(get_spot('BTC', since_minutes=5), n=300)

## Joined view: Kalshi YES ask vs Coinbase spot, sub-second

In [ ]:
# For one window, align Kalshi YES ask with Coinbase BTC spot at every Kalshi tick.
# Useful for studying lead/lag between spot moves and Kalshi reprices.
row, ticks = get_window_with_ticks('BTC')
ws_unix_ms = int(row['window_start_ts']) * 1000
ticks['ts_ms'] = ws_unix_ms + ticks['elapsed_ms']

spot = pd.read_sql('''
    SELECT ts_ms, price
    FROM spot_ticks WHERE coin='BTC' AND ts_ms BETWEEN ? AND ?
    ORDER BY ts_ms
''', con, params=(ws_unix_ms, ws_unix_ms + 900_000))

# Merge-asof: nearest spot price BEFORE each kalshi tick
merged = pd.merge_asof(
    ticks.sort_values('ts_ms'),
    spot.rename(columns={'price': 'btc_spot'}),
    on='ts_ms', direction='backward', tolerance=5000
)
merged['et_time'] = merged['ts_ms'].apply(lambda ms: datetime.fromtimestamp(
    ms/1000, tz=ET).strftime('%H:%M:%S.%f')[:-3])
merged = merged[['et_time','t_sec','yes_ask','no_ask','spread','btc_spot']]
show(merged, n=300)

## Custom queries

Drop SQL into a cell and call `show(pd.read_sql(sql, con))` — the interactive view handles wide tables and big result sets gracefully.

In [ ]:
# Example: per-coin per-side WR over last 500 windows
sql = '''
SELECT
  SUBSTR(ticker, 3, 3) AS coin,
  winner,
  COUNT(*) AS n
FROM windows
WHERE ticker LIKE 'KX%15M%' AND winner IS NOT NULL
  AND id > (SELECT MAX(id) - 500 FROM windows)
GROUP BY coin, winner
ORDER BY coin, winner
'''
show(pd.read_sql(sql, con))

In [ ]:
# Example: spot tick rate per coin over last hour
import time
cutoff = int((time.time() - 3600) * 1000)
sql = '''
SELECT coin, COUNT(*) AS ticks_last_hour,
       MIN(price) AS lo, MAX(price) AS hi, AVG(price) AS avg
FROM spot_ticks WHERE ts_ms > ?
GROUP BY coin
'''
show(pd.read_sql(sql, con, params=(cutoff,)))

## Calibration table — dead-hour filter diff

Compares `~/.calibration_table.beforededhrs.json` (built from all windows)
against `~/.calibration_table.json` (built skipping ET hours 17/20/00/03).

`status` column:
- **DROPPED** — cell qualified before but no longer (dead-hour data was
  inflating its edge past the 4pp threshold)
- **ADDED** — cell didn't qualify before but now does (dead-hour data was
  *masking* a real edge)
- **CHANGED** — cell present in both, edge moved ≥ 0.5pp
- **stable** — moved < 0.5pp

In [ ]:
import json
from pathlib import Path

OLD = Path.home() / '.calibration_table.beforededhrs.json'
NEW = Path.home() / '.calibration_table.json'

if not OLD.exists():
    print('no backup at', OLD, '— rebuild left no diff to show')
elif not NEW.exists():
    print('new table missing at', NEW)
else:
    old = json.loads(OLD.read_text())
    new = json.loads(NEW.read_text())
    print(f'OLD  built_at={old["built_at"]:.0f}  windows={old["windows_scanned"]:,}  cells={len(old["cells"])}')
    print(f'NEW  built_at={new["built_at"]:.0f}  windows={new["windows_scanned"]:,}  cells={len(new["cells"])}')
    print(f'  → {old["windows_scanned"]-new["windows_scanned"]:,} windows excluded by dead-hour filter')
    print()

    oc, nc = old['cells'], new['cells']
    rows = []
    for k in set(oc) | set(nc):
        o, n = oc.get(k), nc.get(k)
        coin, side, tz, bucket = k.split('|')
        rows.append({
            'coin': coin, 'side': side, 'tz': tz, 'bucket': bucket,
            'n_old':    o['n']            if o else None,
            'n_new':    n['n']            if n else None,
            'wr_old':   round(o['wr']*100,1)    if o else None,
            'wr_new':   round(n['wr']*100,1)    if n else None,
            'edge_old': round(o['edge']*100,2)  if o else None,
            'edge_new': round(n['edge']*100,2)  if n else None,
        })
    df = pd.DataFrame(rows)
    df['edge_delta'] = (df['edge_new'].fillna(0) - df['edge_old'].fillna(0)).round(2)

    def _status(r):
        if pd.isna(r['edge_new']) and pd.notna(r['edge_old']): return 'DROPPED'
        if pd.notna(r['edge_new']) and pd.isna(r['edge_old']): return 'ADDED'
        if abs(r['edge_delta']) >= 0.5: return 'CHANGED'
        return 'stable'
    df['status'] = df.apply(_status, axis=1)

    print('status counts:')
    print(df['status'].value_counts().to_string())
    print()

    status_order = {'DROPPED':0,'ADDED':1,'CHANGED':2,'stable':3}
    df['_so'] = df['status'].map(status_order)
    df = (df.sort_values(['_so','edge_delta'], ascending=[True, False],
                         key=lambda s: s.abs() if s.name=='edge_delta' else s)
            .drop(columns='_so'))

    cols = ['status','coin','side','tz','bucket',
            'n_old','n_new','wr_old','wr_new','edge_old','edge_new','edge_delta']
    show(df[cols])

### Calibration diff — visual summary

Run the diff cell above first (so `df` is in the kernel). Charts:
1. status counts — how many cells DROPPED / ADDED / CHANGED / stable
2. scatter `edge_old` vs `edge_new` — diagonal = unchanged, off-diagonal = movers; color by status
3. edge_delta histogram — how big are the shifts
4. coin × T-zone heatmap of edge_delta — where the dead-hour pollution lived
5. by-coin × side stacked bar of statuses

In [ ]:
# Plotly charts comparing old vs new calibration tables.
# Requires `df` from the diff cell above.
try:
    import plotly.express as px
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    import plotly.io as pio
    pio.renderers.default = 'notebook'
except ImportError:
    print('pip install -q plotly first'); raise

if 'df' not in dir():
    print('run the diff cell above first so `df` is loaded')
else:
    # 1. status counts
    counts = df['status'].value_counts().reindex(['DROPPED','ADDED','CHANGED','stable']).fillna(0)
    color_map = {'DROPPED':'#e74c3c','ADDED':'#2ecc71','CHANGED':'#f39c12','stable':'#95a5a6'}
    fig1 = px.bar(x=counts.index, y=counts.values,
                  color=counts.index, color_discrete_map=color_map,
                  text=counts.values.astype(int),
                  title='Cell status after dead-hour filter',
                  labels={'x':'status','y':'# cells'})
    fig1.update_traces(textposition='outside')
    fig1.update_layout(height=380, showlegend=False)
    fig1.show()

    # 2. scatter edge_old vs edge_new
    s = df.copy()
    # for plotting, fill NaN edges with sentinel so DROPPED/ADDED show on axes
    s['edge_old_p'] = s['edge_old'].fillna(0)
    s['edge_new_p'] = s['edge_new'].fillna(0)
    fig2 = px.scatter(s, x='edge_old_p', y='edge_new_p', color='status',
                      color_discrete_map=color_map,
                      hover_data=['coin','side','tz','bucket','n_old','n_new',
                                  'edge_old','edge_new','edge_delta'],
                      title='edge_old vs edge_new (each dot = one cell)',
                      labels={'edge_old_p':'edge OLD (pp)','edge_new_p':'edge NEW (pp)'})
    # diagonal y=x
    lo = min(s['edge_old_p'].min(), s['edge_new_p'].min()) - 1
    hi = max(s['edge_old_p'].max(), s['edge_new_p'].max()) + 1
    fig2.add_shape(type='line', x0=lo, y0=lo, x1=hi, y1=hi,
                   line=dict(color='gray', dash='dash'))
    fig2.add_hline(y=0, line_dash='dot', line_color='lightgray')
    fig2.add_vline(x=0, line_dash='dot', line_color='lightgray')
    fig2.update_layout(height=550)
    fig2.show()

    # 3. edge_delta histogram (only CHANGED + stable, where both edges defined)
    both = df.dropna(subset=['edge_old','edge_new'])
    fig3 = px.histogram(both, x='edge_delta', nbins=40, color='status',
                        color_discrete_map=color_map,
                        title='Edge shift distribution (cells present in both tables)',
                        labels={'edge_delta':'edge_new − edge_old (pp)'})
    fig3.add_vline(x=0, line_dash='dash', line_color='black')
    fig3.update_layout(height=400)
    fig3.show()

    # 4. coin × T-zone heatmap of mean |edge_delta|
    h = df.groupby(['coin','tz'])['edge_delta'].apply(
            lambda s: s.abs().mean() if len(s) else None).reset_index()
    pv = h.pivot(index='coin', columns='tz', values='edge_delta')
    # canonical T-zone order
    tz_order = ['T<60','T60-180','T180-420','T420-720','T720+']
    pv = pv.reindex(columns=[c for c in tz_order if c in pv.columns])
    fig4 = px.imshow(pv, color_continuous_scale='Reds', text_auto='.2f',
                     aspect='auto', range_color=[0, 6],
                     labels={'x':'T-zone','y':'coin','color':'mean |Δedge| (pp)'},
                     title='Where did dead-hour pollution live? (mean |edge_delta| by coin × T-zone)')
    fig4.update_layout(height=320)
    fig4.show()

    # 5. by coin × side stacked bar of statuses
    g = df.groupby(['coin','side','status']).size().reset_index(name='n')
    g['cs'] = g['coin'] + '-' + g['side'].str.upper()
    pivot = g.pivot_table(index='cs', columns='status', values='n', fill_value=0)
    pivot = pivot.reindex(columns=[s for s in ['DROPPED','CHANGED','ADDED','stable']
                                    if s in pivot.columns])
    fig5 = go.Figure()
    for s in pivot.columns:
        fig5.add_trace(go.Bar(name=s, x=pivot.index, y=pivot[s],
                              marker_color=color_map.get(s)))
    fig5.update_layout(barmode='stack', height=420,
                       title='Status breakdown by coin × side',
                       xaxis_title='', yaxis_title='# cells')
    fig5.show()